In [ ]:
import pandas as pd
import numpy as np
import ast
import re

import matplotlib
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from matplotlib import animation
from matplotlib.animation import PillowWriter
from matplotlib.patches import Ellipse, Circle
import seaborn as sns

from IPython.display import HTML

In [ ]:
# read csv with these columns and datatypes
edf = pd.read_parquet(f'artifacts/logs.parquet')

### Animation stepping

In [ ]:
import ast
import re
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from matplotlib.patches import Ellipse, Circle, Polygon


def parse_reward_loc(value):
    """Normalize reward_loc values to a clean list."""
    if value is None:
        return []
    if type(value).__name__ == "NAType":
        return []
    if isinstance(value, (float, np.floating)) and np.isnan(value):
        return []
    if isinstance(value, str):
        raw = value.strip()
        if raw == "" or raw.lower() in {"none", "nan"}:
            return []
        try:
            value = ast.literal_eval(raw)
        except (ValueError, SyntaxError):
            value = [raw]

    if isinstance(value, np.ndarray):
        value = value.tolist()
    if isinstance(value, (list, tuple)):
        return list(value)
    return [value]


def is_collected_flag(value):
    """Interpret collected robustly for bool/int/float/string storage."""
    if value is None:
        return False
    if type(value).__name__ == "NAType":
        return False
    if isinstance(value, (bool, np.bool_)):
        return bool(value)
    if isinstance(value, (int, np.integer)):
        return value != 0
    if isinstance(value, (float, np.floating)):
        return (not np.isnan(value)) and value != 0
    if isinstance(value, str):
        return value.strip().lower() in {"1", "true", "t", "yes", "y"}
    return bool(value)


def get_reward_coord(direction, grid_size):
    """Map reward direction token to board coordinates."""
    if direction == "u":
        return (grid_size // 2, grid_size - 1)
    if direction == "d":
        return (grid_size // 2, 0)
    if direction == "l":
        return (0, grid_size // 2)
    if direction == "r":
        return (grid_size - 1, grid_size // 2)
    return None


def detect_num_agents(columns):
    """Infer agent count from columns like a1x, a2x, ..."""
    agent_nums = [int(re.findall(r"\d+", col)[0]) for col in columns if re.match(r"a\d+x", col)]
    return max(agent_nums) if agent_nums else 0


def build_agent_columns(num_agents):
    """Build position column names for N agents."""
    cols = []
    for i in range(num_agents):
        cols.append(f"a{i+1}x")
        cols.append(f"a{i+1}y")
    return cols


def compute_agents_center(step, num_agents):
    """Compute center point among all agents for heart placement."""
    center_x = sum(step[f"a{i+1}x"] for i in range(num_agents)) / num_agents
    center_y = sum(step[f"a{i+1}y"] for i in range(num_agents)) / num_agents
    return center_x, center_y


def diagnose_heart_transitions(df, regime, start_frame, end_frame, max_rows=40):
    """Print transition diagnostics to explain heart on/off behavior before animation."""
    subdf = df[(df["regime_idx"] == regime) &
               (df["frame_idx"] >= start_frame) &
               (df["frame_idx"] <= end_frame)].sort_values("frame_idx")

    if subdf.empty:
        print("No rows found for the requested regime/frame range.")
        return subdf

    num_agents = detect_num_agents(subdf.columns)
    if num_agents == 0:
        print("No agent columns were detected (a1x, a1y, ...).")
        return subdf

    cols = ["frame_idx"] + build_agent_columns(num_agents) + ["reward_loc", "collected"]
    steps = subdf[cols].to_records(index=False)

    print(f"Diagnostic rows: {len(steps)}")
    print("frame | collected_raw | collected_flag | reward_len | collected_event | reward_cleared | suspicious")

    prev_collected = False
    prev_reward_len = 0
    shown = 0
    suspicious_count = 0

    for step in steps:
        reward_loc = parse_reward_loc(step["reward_loc"])
        reward_len = len(reward_loc)
        collected_flag = is_collected_flag(step["collected"])

        collected_event = collected_flag and (not prev_collected)
        reward_cleared = prev_reward_len > 0 and reward_len == 0

        suspicious = collected_flag and reward_len == 0 and (not collected_event)
        if suspicious:
            suspicious_count += 1

        if collected_event or reward_cleared or suspicious:
            if shown < max_rows:
                print(
                    f"{step['frame_idx']:>5} | {str(step['collected']):>13} | "
                    f"{str(collected_flag):>14} | {reward_len:>10} | "
                    f"{str(collected_event):>15} | {str(reward_cleared):>14} | {str(suspicious):>10}"
                )
                shown += 1

        prev_collected = collected_flag
        prev_reward_len = reward_len

    if shown == 0:
        print("No notable heart transitions found in this range.")
    if suspicious_count > shown:
        print(f"... plus {suspicious_count - shown} additional suspicious rows.")

    return subdf


def animate_simulation_by_df(
    df,
    regime,
    start_frame,
    end_frame,
    grid_size=11,
    interval=100,
    board_padding=0.65,
    heart_hold_frames=1,
    heart_size=1.0,
    heart_y_offset=0.6,
    trim_figure_border=True,
    use_blit=False,
):
    """
    Create an animation for a given regime and frame range that supports multiple agents.

    Parameters:
      df               : DataFrame with columns like a1x, a1y, reward_loc, collected, etc.
      regime           : Regime index to animate.
      start_frame      : Starting frame number (inclusive).
      end_frame        : Ending frame number (inclusive).
      grid_size        : Board size.
      interval         : Frame interval in ms.
      board_padding    : Visual padding around board; lower values reduce outside whitespace.
      heart_hold_frames: Number of frames to keep heart visible after collect event.
      heart_size       : Desired heart size.
      heart_y_offset   : Vertical offset of heart center relative to agent center.
      trim_figure_border: Remove outer subplot margins.
      use_blit         : If True, uses blitting. False is more stable for complex artists.

    Returns:
      ani              : matplotlib.animation.FuncAnimation
    """

    def create_mouse_body(x, y, angle, color):
        """Create a mouse shape from multiple patches."""
        body = Ellipse((x, y), 0.7, 1.1, angle=angle, color=color, alpha=0.95, zorder=3)

        head_x = x + 0.6 * np.cos(np.radians(angle + 90))
        head_y = y + 0.6 * np.sin(np.radians(angle + 90))
        head = Circle((head_x, head_y), 0.36, color=color, alpha=0.95, zorder=4)

        ear_offset = 0.35
        ear1_x = head_x + ear_offset * np.cos(np.radians(angle + 145))
        ear1_y = head_y + ear_offset * np.sin(np.radians(angle + 145))
        ear1 = Circle((ear1_x, ear1_y), 0.18, color=color, alpha=0.95, zorder=4)

        ear2_x = head_x + ear_offset * np.cos(np.radians(angle + 35))
        ear2_y = head_y + ear_offset * np.sin(np.radians(angle + 35))
        ear2 = Circle((ear2_x, ear2_y), 0.18, color=color, alpha=0.95, zorder=4)

        tail_start_x = x - 0.5 * np.cos(np.radians(angle + 90))
        tail_start_y = y - 0.5 * np.sin(np.radians(angle + 90))
        tail_end_x = tail_start_x - 0.7 * np.cos(np.radians(angle + 90))
        tail_end_y = tail_start_y - 0.7 * np.sin(np.radians(angle + 90))
        tail_x = [tail_start_x, tail_end_x]
        tail_y = [tail_start_y, tail_end_y]

        eye_offset = 0.9
        eye1_x = head_x + eye_offset * np.cos(np.radians(angle + 115))
        eye1_y = head_y + eye_offset * np.sin(np.radians(angle + 115))
        eye1 = Circle((eye1_x, eye1_y), 0.05, color="black", alpha=1.0, zorder=5)

        eye2_x = head_x + eye_offset * np.cos(np.radians(angle + 65))
        eye2_y = head_y + eye_offset * np.sin(np.radians(angle + 65))
        eye2 = Circle((eye2_x, eye2_y), 0.05, color="black", alpha=1.0, zorder=5)

        nose_tip_offset = 1.0
        nose_tip_x = head_x + nose_tip_offset * np.cos(np.radians(angle + 90))
        nose_tip_y = head_y + nose_tip_offset * np.sin(np.radians(angle + 90))

        nose_width = 0.08
        nose_base_offset = 0.36
        nose_base_x1 = head_x + nose_base_offset * np.cos(np.radians(angle + 90)) + nose_width * np.cos(np.radians(angle))
        nose_base_y1 = head_y + nose_base_offset * np.sin(np.radians(angle + 90)) + nose_width * np.sin(np.radians(angle))
        nose_base_x2 = head_x + nose_base_offset * np.cos(np.radians(angle + 90)) - nose_width * np.cos(np.radians(angle))
        nose_base_y2 = head_y + nose_base_offset * np.sin(np.radians(angle + 90)) - nose_width * np.sin(np.radians(angle))

        nose = Polygon(
            [(nose_tip_x, nose_tip_y), (nose_base_x1, nose_base_y1), (nose_base_x2, nose_base_y2)],
            color=color,
            alpha=1.0,
            zorder=5,
        )

        return {
            "body": body,
            "head": head,
            "ear1": ear1,
            "ear2": ear2,
            "tail": (tail_x, tail_y),
            "eye1": eye1,
            "eye2": eye2,
            "nose": nose,
        }

    def update_mouse_position(parts, x, y, angle):
        """Update all mouse parts to a new location and orientation."""
        parts["body"].set_center((x, y))
        parts["body"].angle = angle

        head_x = x + 0.6 * np.cos(np.radians(angle + 90))
        head_y = y + 0.6 * np.sin(np.radians(angle + 90))
        parts["head"].set_center((head_x, head_y))

        ear_offset = 0.35
        ear1_x = head_x + ear_offset * np.cos(np.radians(angle + 145))
        ear1_y = head_y + ear_offset * np.sin(np.radians(angle + 145))
        parts["ear1"].set_center((ear1_x, ear1_y))

        ear2_x = head_x + ear_offset * np.cos(np.radians(angle + 35))
        ear2_y = head_y + ear_offset * np.sin(np.radians(angle + 35))
        parts["ear2"].set_center((ear2_x, ear2_y))

        tail_start_x = x - 0.5 * np.cos(np.radians(angle + 90))
        tail_start_y = y - 0.5 * np.sin(np.radians(angle + 90))
        tail_end_x = tail_start_x - 0.7 * np.cos(np.radians(angle + 90))
        tail_end_y = tail_start_y - 0.7 * np.sin(np.radians(angle + 90))
        parts["tail"] = ([tail_start_x, tail_end_x], [tail_start_y, tail_end_y])

        eye_offset = 0.22
        eye1_x = head_x + eye_offset * np.cos(np.radians(angle + 115))
        eye1_y = head_y + eye_offset * np.sin(np.radians(angle + 115))
        parts["eye1"].set_center((eye1_x, eye1_y))

        eye2_x = head_x + eye_offset * np.cos(np.radians(angle + 65))
        eye2_y = head_y + eye_offset * np.sin(np.radians(angle + 65))
        parts["eye2"].set_center((eye2_x, eye2_y))

        nose_tip_offset = 0.5
        nose_tip_x = head_x + nose_tip_offset * np.cos(np.radians(angle + 90))
        nose_tip_y = head_y + nose_tip_offset * np.sin(np.radians(angle + 90))

        nose_width = 0.08
        nose_base_offset = 0.36
        nose_base_x1 = head_x + nose_base_offset * np.cos(np.radians(angle + 90)) + nose_width * np.cos(np.radians(angle))
        nose_base_y1 = head_y + nose_base_offset * np.sin(np.radians(angle + 90)) + nose_width * np.sin(np.radians(angle))
        nose_base_x2 = head_x + nose_base_offset * np.cos(np.radians(angle + 90)) - nose_width * np.cos(np.radians(angle))
        nose_base_y2 = head_y + nose_base_offset * np.sin(np.radians(angle + 90)) - nose_width * np.sin(np.radians(angle))

        parts["nose"].set_xy([(nose_tip_x, nose_tip_y), (nose_base_x1, nose_base_y1), (nose_base_x2, nose_base_y2)])

    def calculate_angle(x_new, y_new, x_old, y_old):
        """Calculate heading from movement direction."""
        dx = x_new - x_old
        dy = y_new - y_old
        if abs(dx) < 0.01 and abs(dy) < 0.01:
            return None
        return np.degrees(np.arctan2(dy, dx)) - 90

    # Filter by regime and frame range.
    subdf = df[(df["regime_idx"] == regime) &
               (df["frame_idx"] >= start_frame) &
               (df["frame_idx"] <= end_frame)].sort_values("frame_idx")

    num_agents = detect_num_agents(subdf.columns)
    if num_agents == 0:
        raise ValueError("No agent position columns found.")

    cols = build_agent_columns(num_agents)
    cols.extend(["reward_loc", "steps_without_reward", "activated", "collected", "terminated", "r1"])
    steps = subdf[cols].to_records(index=False)

    fig, ax = plt.subplots(figsize=(8, 8), dpi=100)
    if trim_figure_border:
        fig.subplots_adjust(left=0, right=1, bottom=0, top=1)

    padding = max(0.0, float(board_padding))
    ax.set_xlim(-0.5 - padding, grid_size - 0.5 + padding)
    ax.set_ylim(-0.5 - padding, grid_size - 0.5 + padding)
    ax.set_aspect("equal")

    ax.set_xticks([])
    ax.set_yticks([])
    ax.grid(False)
    for spine in ax.spines.values():
        spine.set_visible(False)

    ax.set_facecolor("white")
    fig.patch.set_facecolor("white")

    colors = ["#87CEEB", "#FFB6C1", "#90EE90", "#FFD700", "#DDA0DD", "#F0E68C"]
    agent_data = []

    for i in range(num_agents):
        color = colors[i % len(colors)]
        x = steps[0][f"a{i+1}x"]
        y = steps[0][f"a{i+1}y"]
        default_angle = 0

        mouse_parts = create_mouse_body(x, y, default_angle, color)
        ax.add_patch(mouse_parts["body"])
        ax.add_patch(mouse_parts["head"])
        ax.add_patch(mouse_parts["ear1"])
        ax.add_patch(mouse_parts["ear2"])
        ax.add_patch(mouse_parts["eye1"])
        ax.add_patch(mouse_parts["eye2"])
        ax.add_patch(mouse_parts["nose"])

        tail_line, = ax.plot([], [], color=color, linewidth=3, alpha=0.9, zorder=2)

        agent_data.append({
            "parts": mouse_parts,
            "tail_line": tail_line,
            "prev_x": x,
            "prev_y": y,
            "angle": default_angle,
        })

    reward_marker1, = ax.plot([], [], "o", color="#00FF00", alpha=0.6,
                              markersize=20, markeredgewidth=2,
                              markeredgecolor="white", label="Reward", zorder=1)
    reward_marker2, = ax.plot([], [], "o", color="#00FF00", alpha=0.6,
                              markersize=20, markeredgewidth=2,
                              markeredgecolor="white", zorder=1)
    center_marker, = ax.plot([grid_size // 2], [grid_size // 2], "s",
                             color="#FFD700", alpha=0.5, markersize=20,
                             markeredgewidth=2, markeredgecolor="white",
                             label="Center", zorder=1)

    big_heart = ax.fill([], [], color="#FF1493", alpha=0, zorder=11)[0]

    # Precompute heart shape basis; size scaling is applied per-frame.
    t = np.linspace(0, 2 * np.pi, 100)
    heart_base_x = (16 * np.sin(t) ** 3) / 20
    heart_base_y = (13 * np.cos(t) - 5 * np.cos(2 * t) - 2 * np.cos(3 * t) - np.cos(4 * t)) / 20 + heart_y_offset

    hb_x_pos = float(np.max(heart_base_x))
    hb_x_neg = float(abs(np.min(heart_base_x)))
    hb_y_pos = float(np.max(heart_base_y))
    hb_y_neg = float(abs(np.min(heart_base_y)))

    x_min, x_max = ax.get_xlim()
    y_min, y_max = ax.get_ylim()

    def fit_heart_size(cx, cy, desired_size):
        """Shrink heart when near borders so it never clips."""
        limits = [float(desired_size)]

        if hb_x_neg > 0:
            limits.append((cx - x_min) / hb_x_neg)
        if hb_x_pos > 0:
            limits.append((x_max - cx) / hb_x_pos)
        if hb_y_neg > 0:
            limits.append((cy - y_min) / hb_y_neg)
        if hb_y_pos > 0:
            limits.append((y_max - cy) / hb_y_pos)

        return max(0.0, min(limits))

    def build_heart_xy(cx, cy, size):
        return cx + size * heart_base_x, cy + size * heart_base_y

    def init():
        for i, agent in enumerate(agent_data):
            x = steps[0][f"a{i+1}x"]
            y = steps[0][f"a{i+1}y"]
            agent["prev_x"] = x
            agent["prev_y"] = y
            agent["angle"] = 0
            update_mouse_position(agent["parts"], x, y, 0)
            agent["tail_line"].set_data([], [])

        reward_marker1.set_data([], [])
        reward_marker2.set_data([], [])
        center_marker.set_data([grid_size // 2], [grid_size // 2])
        big_heart.set_xy(np.empty((0, 2)))
        big_heart.set_alpha(0)

        ax.heart_frames_left = 0
        ax.heart_center = None
        ax.prev_collected = False

        return [reward_marker1, reward_marker2, center_marker, big_heart]

    def update(frame):
        step = steps[frame]

        for i in range(num_agents):
            x = step[f"a{i+1}x"]
            y = step[f"a{i+1}y"]
            agent = agent_data[i]

            new_angle = calculate_angle(x, y, agent["prev_x"], agent["prev_y"])
            if new_angle is not None:
                angle_diff = new_angle - agent["angle"]
                while angle_diff > 180:
                    angle_diff -= 360
                while angle_diff < -180:
                    angle_diff += 360
                agent["angle"] += angle_diff * 0.8

            update_mouse_position(agent["parts"], x, y, agent["angle"])
            agent["tail_line"].set_data(agent["parts"]["tail"][0], agent["parts"]["tail"][1])
            agent["prev_x"] = x
            agent["prev_y"] = y

        reward_loc = parse_reward_loc(step["reward_loc"])
        collected = is_collected_flag(step["collected"])

        if not hasattr(ax, "heart_frames_left"):
            ax.heart_frames_left = 0
        if not hasattr(ax, "heart_center"):
            ax.heart_center = None
        if not hasattr(ax, "prev_collected"):
            ax.prev_collected = False

        collected_event = collected and (not ax.prev_collected)
        if collected_event:
            ax.heart_center = compute_agents_center(step, num_agents)
            ax.heart_frames_left = max(1, int(heart_hold_frames))

        if frame > 0:
            prev_reward_loc = parse_reward_loc(steps[frame - 1]["reward_loc"])
            reward_cleared = len(prev_reward_loc) > 0 and len(reward_loc) == 0
            if reward_cleared:
                ax.heart_frames_left = 0
                ax.heart_center = None

        if ax.heart_frames_left > 0 and ax.heart_center is not None:
            fitted_size = fit_heart_size(ax.heart_center[0], ax.heart_center[1], heart_size)
            heart_x, heart_y = build_heart_xy(ax.heart_center[0], ax.heart_center[1], fitted_size)
            big_heart.set_xy(np.column_stack([heart_x, heart_y]))
            big_heart.set_alpha(1.0)
            ax.heart_frames_left -= 1
        else:
            big_heart.set_xy(np.empty((0, 2)))
            big_heart.set_alpha(0)

        if len(reward_loc) > 0:
            coord1 = get_reward_coord(reward_loc[0], grid_size)
            if coord1 is not None:
                reward_marker1.set_data([coord1[0]], [coord1[1]])
            else:
                reward_marker1.set_data([], [])

            if len(reward_loc) > 1:
                coord2 = get_reward_coord(reward_loc[1], grid_size)
                if coord2 is not None:
                    reward_marker2.set_data([coord2[0]], [coord2[1]])
                else:
                    reward_marker2.set_data([], [])
            else:
                reward_marker2.set_data([], [])

            center_marker.set_data([], [])
        else:
            reward_marker1.set_data([], [])
            reward_marker2.set_data([], [])
            center_marker.set_data([grid_size // 2], [grid_size // 2])

        ax.set_facecolor("white")
        ax.prev_collected = collected

        return_list = [reward_marker1, reward_marker2, center_marker, big_heart]
        for agent in agent_data:
            return_list.extend([
                agent["parts"]["body"],
                agent["parts"]["head"],
                agent["parts"]["ear1"],
                agent["parts"]["ear2"],
                agent["parts"]["eye1"],
                agent["parts"]["eye2"],
                agent["parts"]["nose"],
                agent["tail_line"],
            ])
        return return_list

    ani = animation.FuncAnimation(
        fig,
        update,
        frames=len(steps),
        init_func=init,
        blit=use_blit,
        interval=interval,
        repeat=True,
    )
    return ani

In [ ]:
diagnose_heart_transitions(edf, regime=1, start_frame=0, end_frame=500, max_rows=60)
ani = animate_simulation_by_df(
    edf,
    regime=1,
    start_frame=0,
    end_frame=500,
    grid_size=11,
    interval=200,
    board_padding=0.65,
    heart_hold_frames=1,
    heart_size=1.0,
    heart_y_offset=0.6,
    trim_figure_border=True,
    use_blit=False,
)
HTML(ani.to_jshtml())

In [ ]:
# # # Save as GIF
gif_filename = "t.gif"
ani.save(gif_filename, writer=PillowWriter(fps=7))

print(f"Animation saved as {gif_filename}")